In [1]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="ylacombe/expresso", repo_type="dataset", local_dir="./expresso")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 14 files: 100%|██████████| 14/14 [00:05<00:00,  2.62it/s]


'/home/ubuntu/expresso'

In [2]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
files = glob('expresso/read/*.parquet')
len(files)

12

In [9]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': df['text'].iloc[i],
                'speaker': f"{base}_{df['speaker_id'].iloc[i]}_{df['style'].iloc[i]}"
            })
        
    return data

In [10]:
data = loop((files[:1], 0))

100%|██████████| 1/1 [00:49<00:00, 49.75s/it]


In [12]:
len(data)

968

In [15]:
data = multiprocessing(files, loop, len(files))

100%|██████████| 1/1 [00:57<00:00, 57.87s/it]


In [16]:
len(data)

11615

In [17]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'expresso_audio/expresso-read-train-00000-of-00012_0.mp3',
 'text': 'Why are you beating up my jukebox?',
 'speaker': 'expresso_audio_ex01_confused'}

In [18]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'expresso')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 261.23ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  201kB /  201kB,  608kB/s  
Processing Files (1 / 1): 100%|██████████|  201kB /  201kB,  502kB/s  
New Data Upload: 100%|██████████|  201kB /  201kB,  502kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.34 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/6a973bc82b4f6968792d29defd2eb1ae534375e1', commit_message='Upload dataset', commit_description='', oid='6a973bc82b4f6968792d29defd2eb1ae534375e1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [19]:
audio_files = [d['audio_filename'] for d in data]

with open('expresso-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [37]:
folders = glob('expresso_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

expresso_audio_neucodec
expresso_audio


In [38]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('expresso_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  10%|▉         | 40.4MB /  413MB,   ???B/s  
Processing Files (0 / 1):  43%|████▎     |  177MB /  413MB,  682MB/s  
Processing Files (0 / 1):  77%|███████▋  |  319MB /  413MB,  698MB/s  
Processing Files (0 / 1):  99%|█████████▉|  410MB /  413MB,  617MB/s  
Processing Files (0 / 1):  99%|█████████▉|  411MB /  413MB,  463MB/s  
Processing Files (0 / 1): 100%|█████████▉|  411MB /  413MB,  371MB/s  
Processing Files (0 / 1): 100%|█████████▉|  412MB /  413MB,  310MB/s  
Processing Files (1 / 1): 100%|██████████|  413MB /  413MB,  266MB/s  
Processing Files (1 / 1): 100%|██████████|  413MB /  413MB,  233MB/s  
New Data Upload: 100%|██████████|  413MB /  413MB,  233MB/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  99%|█████████▊| 8.96MB / 9.09MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 9.09MB / 9.09MB,  815kB/s  
Processing Files (1 / 1